# LayoutExtractor (Docling) Test Notebook

This notebook demonstrates and tests the `LayoutExtractor` strategy (Strategy B)
which uses **Docling** for layout-aware extraction:

- Multi-column text handling
- Table structure preservation
- Figures/images with captions
- Normalisation into the shared `ExtractedDocument` schema.


In [ ]:
import os
import shutil

# Make sure HF uses our non-symlink implementation
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "true"  # extra belt-and-suspenders
os.environ["HF_HOME"] = r"C:\hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# Monkeypatch Hugging Face hub to avoid os.symlink on Windows
import huggingface_hub.file_download as fd

def _create_symlink_noop(src: str, dst: str, new_blob: bool):
    """
    Replacement for huggingface_hub.file_download._create_symlink on Windows.

    Instead of creating a symlink, it just copies the file.
    """
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

fd._create_symlink = _create_symlink_noop

c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
## Setup and Imports

import sys
from pathlib import Path

import os


# Add src to path
sys.path.insert(0, str(Path().absolute().parent))

from src.agents.triage import TriageAgent
from src.strategies.layout_aware import LayoutExtractor
from src.models.document_profile import DocumentProfile


In [ ]:
## Select Test Document

# Reuse the same corpus paths as the fast text notebook
TEST_DOCS = {
    "class_a": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_a\CBE Annual Report 2006-7 .pdf"),
    "class_b": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_b\Annual_Report_JUNE-2023.pdf"),
    "class_c": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_c\fta_performance_survey_final_report_2022.pdf"),
    "class_d": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_d\tax_expenditure_ethiopia_2021_22.pdf"),
}

selected_doc = "class_a"  # change to b/c/d to test other classes
pdf_path = TEST_DOCS[selected_doc]

if not pdf_path.exists():
    print(f"⚠️ Document not found: {pdf_path}")
else:
    print(f"✓ Selected document: {pdf_path.name}")
    print(f"  Path: {pdf_path}")
    print(f"  Size: {pdf_path.stat().st_size / 1024 / 1024:.2f} MB")


✓ Selected document: CBE Annual Report 2006-7 .pdf
  Path: C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_a\CBE Annual Report 2006-7 .pdf
  Size: 34.17 MB


In [ ]:
## Step 1: Triage and Profile

profiles_dir = Path(".refinery/profiles")
profiles_dir.mkdir(parents=True, exist_ok=True)
triage = TriageAgent(profiles_dir=profiles_dir)

print("🔍 Running Triage Agent...")
profile = triage.classify_document(pdf_path)

print("\n📋 Document Profile (summary):")
print(f"  - Document ID: {profile.doc_id}")
print(f"  - Origin: {profile.origin_type}")
print(f"  - Layout: {profile.layout_complexity}")
print(f"  - Domain: {profile.domain_hint}")
print(f"  - Estimated Cost: {profile.estimated_cost}")
print(f"  - Language: {profile.language} (confidence: {profile.language_confidence:.2f})")
print(f"  - Pages: {profile.metadata.page_count}")
print(f"  - Size: {profile.metadata.size_bytes / 1024 / 1024:.2f} MB")


🔍 Running Triage Agent...

📋 Document Profile (summary):
  - Document ID: CBE Annual Report 2006-7 
  - Origin: mixed
  - Layout: table_heavy
  - Domain: financial
  - Estimated Cost: needs_layout_model
  - Language: en (confidence: 1.00)
  - Pages: 60
  - Size: 34.17 MB


In [ ]:
## Step 2: Initialise LayoutExtractor

try:
    layout_extractor = LayoutExtractor()
    print("✓ Initialised LayoutExtractor (Docling)")
except ImportError as e:
    print("✗ Could not initialise LayoutExtractor:", e)
    print("  Make sure docling is installed: pip install docling")
    raise

print("\n🔍 Checking if LayoutExtractor can handle this profile...")
can_handle = layout_extractor.can_handle(profile)
print(f"  - Can Handle: {can_handle}")


✓ Initialised LayoutExtractor (Docling)

🔍 Checking if LayoutExtractor can handle this profile...
  - Can Handle: True


In [ ]:
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions

# Optional: use the faster v2 backend if available
try:
    from docling.backend.docling_parse_v2_backend import DoclingParseV2DocumentBackend
    BACKEND = DoclingParseV2DocumentBackend
except Exception:
    from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
    BACKEND = PyPdfiumDocumentBackend


def create_fast_docling_converter(use_ocr: bool) -> DocumentConverter:
    """Create a Docling converter tuned for speed on digital PDFs.

    Args:
        use_ocr: Whether to enable OCR (True only for scanned-image PDFs).
    """
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = use_ocr           # True only for scanned docs
    pipeline_options.do_table_structure = True  # Keep if you need tables

    # You can turn off some extra features for more speed, e.g.:
    # pipeline_options.do_cell_matching = False

    # Newer Docling versions let you pass pipeline_options and backend directly
    return DocumentConverter(
        pipeline_options=pipeline_options,
        backend=BACKEND,
    )


# Decide based on triage profile
use_ocr = profile.origin_type == "scanned_image"
fast_converter = create_fast_docling_converter(use_ocr=use_ocr)

# Override the internal converter of LayoutExtractor JUST for this notebook
layout_extractor._converter = fast_converter

print(f"Using Docling with do_ocr={use_ocr}, backend={BACKEND.__name__}")

TypeError: DocumentConverter.__init__() got an unexpected keyword argument 'pipeline_options'

In [ ]:
## Step 3: Confidence and Cost

print("📊 Calculating layout-aware confidence score...")
layout_conf = layout_extractor.confidence_score(str(pdf_path))
print(f"  - Confidence: {layout_conf:.4f} ({layout_conf*100:.1f}%)")

print("\n💰 Estimating layout-aware extraction cost...")
layout_cost = layout_extractor.cost_estimate(str(pdf_path))
print(f"  - Total Cost: ${layout_cost['total_cost_usd']:.4f}")
print(f"  - Cost per Page: ${layout_cost['cost_per_page']:.4f}")


📊 Calculating layout-aware confidence score...
  - Confidence: 0.6096 (61.0%)

💰 Estimating layout-aware extraction cost...
  - Total Cost: $0.0300
  - Cost per Page: $0.0005


In [ ]:
## Step 4: Run Layout-Aware Extraction

print("📄 Extracting with LayoutExtractor (Docling)...")

import time
start_time = time.time()

extracted = layout_extractor.extract(str(pdf_path))

elapsed = time.time() - start_time
print(f"\n✓ Extraction completed in {elapsed:.2f} seconds")
print("\n📊 Extraction Summary (Docling → ExtractedDocument):")
print(f"  - Text Blocks: {len(extracted.text_blocks):,}")
print(f"  - Tables: {len(extracted.tables)}")
print(f"  - Figures: {len(extracted.figures)}")
print(f"  - Reading Order indices: {len(extracted.reading_order)}")


📄 Extracting with LayoutExtractor (Docling)...


[INFO] 2026-03-05 13:48:36,980 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-05 13:48:37,079 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-05 13:48:37,082 [RapidOCR] main.py:53: Using C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-05 13:48:37,495 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-05 13:48:37,507 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-05 13:48:37,511 [RapidOCR] main.py:53: Using C:\Users\Hello\OneDrive\Desktop\Tenaciou

In [ ]:
## Step 5: Inspect Sample Tables and Figures

print("📊 Sample Tables (first 3):")
print("=" * 80)
for i, table in enumerate(extracted.tables[:3], start=1):
    print(f"\n[Table {i}] Page {table.page_num}")
    print(f"  Headers: {table.headers}")
    if table.rows:
        print(f"  First row: {table.rows[0]}")

print("\n🖼️ Sample Figures (first 3):")
print("=" * 80)
for i, fig in enumerate(extracted.figures[:3], start=1):
    print(f"\n[Figure {i}] Page {fig.page_num}")
    print(f"  Caption: {fig.caption or '(none)'}")
    print(f"  BBox: ({fig.bbox.x0:.1f}, {fig.bbox.y0:.1f}) → ({fig.bbox.x1:.1f}, {fig.bbox.y1:.1f})")


📊 Sample Tables (first 3):


NameError: name 'extracted' is not defined

In [ ]:
## Step 6: Summary

print("✅ LayoutExtractor (Docling) Test Summary:")
print("=" * 80)
print(f"\nDocument: {pdf_path.name}")
print("\nProfile:")
print(f"  - Origin: {profile.origin_type}")
print(f"  - Layout: {profile.layout_complexity}")
print(f"  - Domain: {profile.domain_hint}")
print("\nStrategy:")
print(f"  - Strategy: {layout_extractor.name}")
print(f"  - Can Handle: {can_handle}")
print(f"  - Confidence: {layout_conf:.2%}")
print(f"  - Estimated Cost: ${layout_cost['total_cost_usd']:.4f}")
print("\nExtraction Results:")
print(f"  - Text Blocks: {len(extracted.text_blocks):,}")
print(f"  - Tables: {len(extracted.tables)}")
print(f"  - Figures: {len(extracted.figures)}")


✅ LayoutExtractor (Docling) Test Summary:

Document: fta_performance_survey_final_report_2022.pdf

Profile:
  - Origin: native_digital
  - Layout: table_heavy
  - Domain: technical

Strategy:
  - Strategy: layout_aware
  - Can Handle: True


NameError: name 'layout_conf' is not defined

In [ ]:
## Step 4a: Convert to Markdown First

print("📝 Converting document to Markdown...")

import time
start_time = time.time()

# Convert using Docling
result = layout_extractor._converter.convert(str(pdf_path))
docling_document = getattr(result, "document", result)

# Export to Markdown
markdown_content = docling_document.export_to_markdown()

elapsed = time.time() - start_time
print(f"✓ Markdown conversion completed in {elapsed:.2f} seconds")

# Save Markdown file
markdown_path = pdf_path.with_suffix('.md')
with open(markdown_path, 'w', encoding='utf-8') as f:
    f.write(markdown_content)

print(f"✓ Markdown saved to: {markdown_path}")
print(f"  - File size: {len(markdown_content):,} characters")
print(f"  - Lines: {len(markdown_content.splitlines()):,}")

# Show preview
print("\n📄 Markdown Preview (first 500 characters):")
print("=" * 80)
print(markdown_content[:500])
print("...")

In [ ]:
## Step 4b: Analyze Markdown Content

import re
from collections import Counter

print("🔍 Analyzing Markdown structure...\n")

# Count different Markdown elements
analysis = {
    "headings": {
        "h1": len(re.findall(r'^#\s+', markdown_content, re.MULTILINE)),
        "h2": len(re.findall(r'^##\s+', markdown_content, re.MULTILINE)),
        "h3": len(re.findall(r'^###\s+', markdown_content, re.MULTILINE)),
        "h4": len(re.findall(r'^####\s+', markdown_content, re.MULTILINE)),
    },
    "tables": len(re.findall(r'^\|.*\|', markdown_content, re.MULTILINE)) // 3,  # Rough estimate
    "code_blocks": len(re.findall(r'```', markdown_content)) // 2,
    "links": len(re.findall(r'\[.*?\]\(.*?\)', markdown_content)),
    "images": len(re.findall(r'!\[.*?\]\(.*?\)', markdown_content)),
    "lists": {
        "unordered": len(re.findall(r'^[\*\-\+]\s+', markdown_content, re.MULTILINE)),
        "ordered": len(re.findall(r'^\d+\.\s+', markdown_content, re.MULTILINE)),
    },
    "bold": len(re.findall(r'\*\*.*?\*\*', markdown_content)),
    "italic": len(re.findall(r'\*.*?\*', markdown_content)) - len(re.findall(r'\*\*.*?\*\*', markdown_content)),
}

print("📊 Markdown Structure Analysis:")
print("=" * 80)
print(f"Headings:")
print(f"  - H1 (main sections): {analysis['headings']['h1']}")
print(f"  - H2 (subsections): {analysis['headings']['h2']}")
print(f"  - H3 (sub-subsections): {analysis['headings']['h3']}")
print(f"  - H4 (deep subsections): {analysis['headings']['h4']}")
print(f"\nTables: ~{analysis['tables']} (estimated from pipe patterns)")
print(f"Code blocks: {analysis['code_blocks']}")
print(f"Links: {analysis['links']}")
print(f"Images: {analysis['images']}")
print(f"Lists:")
print(f"  - Unordered: {analysis['lists']['unordered']}")
print(f"  - Ordered: {analysis['lists']['ordered']}")
print(f"Bold text: {analysis['bold']} instances")
print(f"Italic text: {analysis['italic']} instances")

# Extract heading structure
headings = re.findall(r'^(#{1,4})\s+(.+)$', markdown_content, re.MULTILINE)
if headings:
    print(f"\n📑 Document Outline (first 10 headings):")
    print("=" * 80)
    for level, title in headings[:10]:
        indent = "  " * (len(level) - 1)
        print(f"{indent}{level} {title}")

# Find table structures
table_pattern = r'\|(.+)\|\n\|[-\s\|]+\|\n((?:\|.+\|\n?)+)'
tables = re.findall(table_pattern, markdown_content)
if tables:
    print(f"\n📊 Sample Table (first table found):")
    print("=" * 80)
    headers, rows = tables[0]
    print(f"Headers: {headers}")
    print(f"Rows: {len(rows.split('|')) - 2} columns detected")
    print(f"First few rows:\n{rows[:200]}...")